# Old School RuneScape — Level 99 skill statistics (EDA)

Dataset: [samanthakamal33122/old-school-runescape-level-99-skill-dataset](https://www.kaggle.com/datasets/samanthakamal33122/old-school-runescape-level-99-skill-dataset) (CC0-1.0).

Each row is **one skill** (23 skills) with aggregate player counts and quest / XP metadata from **January 2023**.

- **lvl_99_users** / **lvl_99_user_na**: how many players have 99 in this skill vs. 99 here but not max in every skill.
- **xp_max_users**: players who hit the XP cap in this skill.
- **qst_\***, **max_\***: quest level/XP requirements tied to the skill.
- **mini_xp** / **max_xp_mini**: minigame XP hooks (sparse in this extract).

**Correlation note:** With one row per skill, “correlation between skills” is not a player×skill matrix; we report **correlations between numeric columns across skills** (which metrics rise and fall together), plus **Spearman rank** as a robust check on small *n* = 23.

## Setup & load

In [1]:
import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path
from IPython.display import display, HTML

PRISM = list(px.colors.qualitative.Prism)
px.defaults.color_discrete_sequence = PRISM
px.defaults.color_continuous_scale = [
    [i / (len(PRISM) - 1), c] for i, c in enumerate(PRISM)
]

_IS_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or Path("/kaggle").exists()
if _IS_KAGGLE:
    pio.renderers.default = "iframe"


def show_plotly(fig):
    if os.environ.get("PLOTLY_FORCE_HTML", "").lower() in ("1", "true", "yes"):
        display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))
    elif _IS_KAGGLE:
        fig.show()
    else:
        display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))


CSV_NAME = "OSRS Skill XP Data Jan 2023 - Sheet1.csv"
KAGGLE_PATHS = [
    Path(f"/kaggle/input/datasets/samanthakamal33122/old-school-runescape-level-99-skill-dataset/{CSV_NAME}"),
]
LOCAL_PATH = Path("data") / CSV_NAME

csv_path = next((p for p in KAGGLE_PATHS if p.exists()), None)
if csv_path is not None:
    print(f"Using Kaggle path: {csv_path}")
else:
    csv_path = LOCAL_PATH if LOCAL_PATH.exists() else Path.cwd() / "data" / CSV_NAME
    print(f"Using local path: {csv_path}")

df = pd.read_csv(csv_path, thousands=",")
assert len(df) == 23, f"expected 23 skills, got {len(df)}"
df.head()

Using local path: data/OSRS Skill XP Data Jan 2023 - Sheet1.csv


,Skill,lvl_99_users,lvl_99_user_na,xp_max_users,qst_lvl_req,max_lvl_need,qst_xp_req,max_xp_req,qst_xp_noreq,max_xp_noreq,mini_xp,max_xp_mini
0,Strength,433113,401946,623,3,50,11,22000,1,13750.0,1,20000.0
1,Hitpoints,410160,378993,467,1,50,9,25000,2,6325.0,1,20000.0
2,Ranged,377347,346180,538,8,60,7,10500,0,NaN,0,NaN
3,Cooking,292048,260881,2192,11,70,10,10000,5,1525.0,0,NaN
4,Magic,282334,251167,231,17,75,13,20000,3,1000.0,0,NaN


## Column overview

In [2]:
df.info()
display(df.describe().T)
df.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 23 entries, 0 to 22
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Skill           23 non-null     str    
 1   lvl_99_users    23 non-null     int64  
 2   lvl_99_user_na  23 non-null     int64  
 3   xp_max_users    23 non-null     int64  
 4   qst_lvl_req     23 non-null     int64  
 5   max_lvl_need    23 non-null     int64  
 6   qst_xp_req      23 non-null     int64  
 7   max_xp_req      23 non-null     int64  
 8   qst_xp_noreq    23 non-null     int64  
 9   max_xp_noreq    18 non-null     float64
 10  mini_xp         23 non-null     int64  
 11  max_xp_mini     4 non-null      float64
dtypes: float64(2), int64(9), str(1)
memory usage: 2.3 KB


,count,mean,std,min,25%,50%,75%,max
lvl_99_users,23.0,175595.695652,119954.558239,53103.0,71343.5,124774.0,259411.00,433113.0
lvl_99_user_na,23.0,144428.695652,119954.558239,21936.0,40176.5,93607.0,228244.00,401946.0
xp_max_users,23.0,406.739130,429.635492,87.0,183.0,321.0,502.50,2192.0
qst_lvl_req,23.0,12.000000,9.214613,1.0,5.5,10.0,15.50,36.0
max_lvl_need,23.0,64.304348,8.120875,50.0,60.0,69.0,70.00,75.0
qst_xp_req,23.0,10.652174,4.886053,5.0,8.0,9.0,11.50,25.0
max_xp_req,23.0,39810.869565,20404.489014,8000.0,23500.0,40000.0,50000.00,80000.0
qst_xp_noreq,23.0,2.260870,2.240483,0.0,1.0,2.0,3.50,8.0
max_xp_noreq,18.0,6782.277778,13911.720530,250.0,1000.0,2125.0,5743.75,60000.0
mini_xp,23.0,0.173913,0.387553,0.0,0.0,0.0,0.00,1.0


Skill              0
lvl_99_users       0
lvl_99_user_na     0
xp_max_users       0
qst_lvl_req        0
max_lvl_need       0
qst_xp_req         0
max_xp_req         0
qst_xp_noreq       0
max_xp_noreq       5
mini_xp            0
max_xp_mini       19
dtype: int64

## Skill distribution — popularity (level 99 counts)

Sorted by **lvl_99_users** (how many players achieved 99 in that skill).

In [3]:
pop = df.sort_values("lvl_99_users", ascending=True)
fig = px.bar(
    pop,
    x="lvl_99_users",
    y="Skill",
    title="Players with level 99 per skill (Jan 2023)",
    color="lvl_99_users",
    color_continuous_scale=px.defaults.color_continuous_scale,
)
fig.update_layout(yaxis={"categoryorder": "total ascending"}, showlegend=False, height=640)
show_plotly(fig)

In [4]:
df2 = df.assign(
    pct_not_all_99=lambda d: (d["lvl_99_user_na"] / d["lvl_99_users"].replace(0, pd.NA)) * 100
)
fig = px.bar(
    df2.sort_values("pct_not_all_99", ascending=True),
    x="pct_not_all_99",
    y="Skill",
    title="Among 99s: % who do not have 99 in every skill (higher = more 'specialist' 99s?)",
    labels={"pct_not_all_99": "% of lvl_99_users"},
)
fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=640)
show_plotly(fig)

## Quest & XP requirement columns (per skill)

Facets include **xp_max_users**, quest requirement counts (**qst_lvl_req**, **max_lvl_need**, **qst_xp_req**, **qst_xp_noreq**), XP reward caps (**max_xp_req**, **max_xp_noreq**), and minigame flags (**mini_xp**, **max_xp_mini**). Plotly’s default `facet_col` **links every subplot’s y-axis to the same scale** — with **max_xp_req** in the tens of thousands, smaller integer counts look empty until we set **independent y** per facet (`update_yaxes(matches=None)`).

In [5]:
skill_order = df.sort_values("lvl_99_users")["Skill"].tolist()
melt_cols = [
    "xp_max_users",
    "qst_lvl_req",
    "max_lvl_need",
    "qst_xp_req",
    "max_xp_req",
    "qst_xp_noreq",
    "max_xp_noreq",
    "mini_xp",
    "max_xp_mini",
]
long = df.melt(id_vars=["Skill"], value_vars=melt_cols, var_name="metric", value_name="value")
fig = px.bar(
    long.sort_values(["metric", "value"]),
    x="Skill",
    y="value",
    facet_col="metric",
    facet_col_wrap=3,
    title="Per-skill quest / XP / minigame stats (independent y per facet)",
    category_orders={"Skill": skill_order},
)
fig.update_xaxes(tickangle=-45)
# Default: y2,y3,… match y → one global ymax (~80k from max_xp_req) squashes qst_* bars to ~0 height.
fig.update_yaxes(matches=None)
fig.update_layout(height=1100, showlegend=False)
show_plotly(fig)

## Correlations between numeric features

Pearson and Spearman on **all numeric columns** (pairwise deletion for missing **max_xp_noreq** / **max_xp_mini**). *n* = 23 rows — treat large |r| as suggestive, not definitive.

In [6]:
num = df.select_dtypes(include=["number"])
pear = num.corr(method="pearson")
spear = num.corr(method="spearman")

fig = go.Figure(
    data=go.Heatmap(
        z=pear.values,
        x=pear.columns,
        y=pear.columns,
        zmin=-1,
        zmax=1,
        colorscale="RdBu",
        reversescale=True,
        text=pear.round(2).values,
        texttemplate="%{text}",
        hovertemplate="%{y} vs %{x}<br>r=%{z:.3f}<extra></extra>",
    )
)
fig.update_layout(title="Pearson correlation matrix (across 23 skills)", height=720, width=900)
show_plotly(fig)

fig2 = go.Figure(
    data=go.Heatmap(
        z=spear.values,
        x=spear.columns,
        y=spear.columns,
        zmin=-1,
        zmax=1,
        colorscale="RdBu",
        reversescale=True,
        text=spear.round(2).values,
        texttemplate="%{text}",
    )
)
fig2.update_layout(title="Spearman rank correlation matrix", height=720, width=900)
show_plotly(fig2)

In [7]:
def top_corr_pairs(corr: pd.DataFrame, k: int = 12):
    c = corr.copy()
    v = c.to_numpy().copy()
    v[np.tril_indices_from(v)] = np.nan
    s = pd.DataFrame(v, index=c.index, columns=c.columns).stack().dropna()
    return s.abs().sort_values(ascending=False).head(k)


print("Top |Pearson| pairs (excluding diagonal):")
display(top_corr_pairs(pear))
print("Top |Spearman| pairs:")
display(top_corr_pairs(spear))

Top |Pearson| pairs (excluding diagonal):


lvl_99_users    lvl_99_user_na    1.000000
qst_lvl_req     qst_xp_req        0.841992
lvl_99_user_na  max_xp_req        0.681914
lvl_99_users    max_xp_req        0.681914
lvl_99_user_na  mini_xp           0.645035
lvl_99_users    mini_xp           0.645035
max_lvl_need    mini_xp           0.609725
qst_xp_req      qst_xp_noreq      0.569211
qst_lvl_req     max_lvl_need      0.549726
lvl_99_users    max_lvl_need      0.473470
lvl_99_user_na  max_lvl_need      0.473470
max_lvl_need    max_xp_req        0.458715
dtype: float64

Top |Spearman| pairs:


lvl_99_users    lvl_99_user_na    1.000000
qst_lvl_req     max_lvl_need      0.707124
lvl_99_user_na  xp_max_users      0.684952
lvl_99_users    xp_max_users      0.684952
lvl_99_user_na  max_xp_req        0.671651
lvl_99_users    max_xp_req        0.671651
qst_lvl_req     mini_xp           0.657942
lvl_99_users    mini_xp           0.553372
lvl_99_user_na  mini_xp           0.553372
max_lvl_need    mini_xp           0.540264
qst_lvl_req     max_xp_req        0.537836
                qst_xp_req        0.528394
dtype: float64

In [8]:
scatter = df.copy()
fig = px.scatter(
    scatter,
    x="lvl_99_users",
    y="lvl_99_user_na",
    text="Skill",
    title="Strong expected link: 99 cape counts vs. 'not all 99s' subset",
    trendline="ols",
)
fig.update_traces(textposition="top center")
fig.update_layout(height=520)
show_plotly(fig)

In [9]:
fig = px.scatter(
    scatter,
    x="lvl_99_users",
    y="xp_max_users",
    text="Skill",
    title="Popularity vs. max-XP players (log y helps heavy tail)",
    log_y=True,
    trendline="ols",
)
fig.update_traces(textposition="top center")
fig.update_layout(height=520)
show_plotly(fig)